# Full fine-tuning of an encoder model for text classification
Fine-tunes a BERT-style encoder (DistilBERT, BERT, RoBERTa, DeBERTa…) end to end for single-label
text classification — intent detection, sentiment, ticket routing, topic labelling — with the
Hugging Face `Trainer`. Reports accuracy and macro-F1 on a held-out split; metrics go to MLflow.

Data: CSV/JSONL with a text column and a label column (string or integer labels), or a Hugging
Face dataset id. Output: the fine-tuned model + tokenizer, label mapping and metrics.

Requires: torch, transformers, datasets, accelerate. A GPU is recommended.

## Parameters

In [ ]:
base_model = "distilbert-base-uncased"   # encoder checkpoint
dataset_path = ""            # CSV/JSONL file (staged by the platform) - or use hf_dataset
hf_dataset = ""              # e.g. "imdb"
hf_split = "train"
text_column = "text"
label_column = "label"
num_train_epochs = 3
learning_rate = 5e-5
batch_size = 16
max_length = 256
eval_fraction = 0.1
fp16 = True
seed = 42
output_dir = "outputs"

## Setup

In [ ]:
import json, os
output_dir = os.environ.get("STUDIO_OUTPUT_DIR", output_dir) if output_dir == "outputs" else output_dir
os.makedirs(output_dir, exist_ok=True)
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
set_seed(seed)
report_to = ["mlflow"] if os.environ.get("MLFLOW_RUN_ID") else []
print("cuda:", torch.cuda.is_available())

## Load data and labels

In [ ]:
if dataset_path:
    fmt = "csv" if dataset_path.endswith(".csv") else "json"
    ds = load_dataset(fmt, data_files=dataset_path, split="train")
elif hf_dataset:
    ds = load_dataset(hf_dataset, split=hf_split)
else:
    raise ValueError("Set dataset_path or hf_dataset")
labels = sorted({str(x) for x in ds[label_column]})
label2id = {l: i for i, l in enumerate(labels)}
ds = ds.map(lambda ex: {"labels": label2id[str(ex[label_column])]})
split = ds.train_test_split(test_size=eval_fraction, seed=seed, stratify_by_column=None)
print(len(labels), "labels:", labels[:20])

## Tokenize

In [ ]:
tok = AutoTokenizer.from_pretrained(base_model)
cols = [c for c in ds.column_names if c != "labels"]
enc = split.map(lambda b: tok(b[text_column], truncation=True, max_length=max_length), batched=True,
                remove_columns=cols)

## Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    base_model, num_labels=len(labels), id2label={i: l for l, i in label2id.items()}, label2id=label2id)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    acc = float((preds == p.label_ids).mean())
    f1s = []
    for c in range(len(labels)):
        tp = int(((preds == c) & (p.label_ids == c)).sum()); fp = int(((preds == c) & (p.label_ids != c)).sum())
        fn = int(((preds != c) & (p.label_ids == c)).sum())
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return {"accuracy": acc, "f1_macro": float(np.mean(f1s))}

args = TrainingArguments(output_dir=os.path.join(output_dir, "checkpoints"), num_train_epochs=num_train_epochs,
                         learning_rate=learning_rate, per_device_train_batch_size=batch_size,
                         per_device_eval_batch_size=batch_size, eval_strategy="epoch", save_strategy="no",
                         logging_steps=20, fp16=fp16 and torch.cuda.is_available(), report_to=report_to, seed=seed)
trainer = Trainer(model=model, args=args, train_dataset=enc["train"], eval_dataset=enc["test"],
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
train_result = trainer.train()
trainer.save_metrics("train", train_result.metrics)

## Evaluate and save

In [ ]:
ev = trainer.evaluate()
trainer.save_metrics("eval", ev)
print(ev)
model_dir = os.path.join(output_dir, "model")
trainer.save_model(model_dir)
tok.save_pretrained(model_dir)
with open(os.path.join(output_dir, "metrics.json"), "w") as f:
    json.dump({k: v for k, v in ev.items() if isinstance(v, (int, float))}, f, indent=1)
import shutil; shutil.rmtree(os.path.join(output_dir, "checkpoints"), ignore_errors=True)
print("saved to", model_dir)